In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
catalog = "workspace"

source_schema = "default"
source_table = "silver_waffle"

target_schema = "gold"
target_table = "gold_waffle"

key_cols = ["Columns"]
surrogate_key = "ColumnKey"
cdc_col = "modifiedDate"

full_source = f"{source_schema}.{source_table}"
full_target = f"{catalog}.{target_schema}.{target_table}"   

In [0]:
target_exists = spark.catalog.tableExists(full_target)

if target_exists:

    last_load_row = spark.sql(f"""
        SELECT max({cdc_col}) as max_val
        FROM {full_target}
    """).collect()[0]

    last_load = last_load_row["max_val"]

else:
    last_load = None

In [0]:
if last_load:
    df_src = spark.sql(f"""
        SELECT *
        FROM {full_source}
        WHERE {cdc_col} >= TIMESTAMP '{last_load}'
    """)
else:
    df_src = spark.table(full_source)

In [0]:
if len(df_src.take(1)) == 0:
    print("İşlenecek veri yok.")
    dbutils.notebook.exit("No data")

In [0]:
if target_exists:

    select_cols = key_cols + [
        surrogate_key,
        "create_date",
        "update_date",
        cdc_col
    ]

    df_trg = spark.table(full_target).select(*select_cols)

else:
    df_trg = None

In [0]:
if target_exists:

    join_cond = [df_src[k] == df_trg[k] for k in key_cols]

    df_join = df_src.alias("src") \
        .join(df_trg.alias("trg"), join_cond, "left")

    df_existing = df_join.filter(F.col(surrogate_key).isNotNull())
    df_new = df_join.filter(F.col(surrogate_key).isNull())

else:
    df_existing = spark.createDataFrame([], df_src.schema)
    df_new = df_src

In [0]:
df_existing_final = df_existing \
    .withColumn("update_date", F.current_timestamp())

In [0]:
if len(df_new.take(1)) > 0:

    if target_exists:

        max_key_row = spark.sql(f"""
            SELECT max({surrogate_key}) as max_key
            FROM {full_target}
        """).collect()[0]

        max_key = max_key_row["max_key"] if max_key_row["max_key"] else 0

    else:
        max_key = 0

    window_spec = Window.orderBy(*key_cols)

    df_new = df_new \
        .withColumn("row_num", F.row_number().over(window_spec)) \
        .withColumn(surrogate_key, F.lit(max_key) + F.col("row_num")) \
        .drop("row_num")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_new_final = df_new \
    .withColumn("create_date", F.current_timestamp()) \
    .withColumn("update_date", F.current_timestamp())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_final = df_existing_final.unionByName(df_new_final)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
if target_exists:

    join_cond = [df_src[k] == df_trg[k] for k in key_cols]

    df_join = df_src.alias("src") \
        .join(df_trg.alias("trg"), join_cond, "left")

    # SADECE gerekli kolonları seç
    df_join = df_join.select(
        "src.*",
        F.col(f"trg.{surrogate_key}"),
        F.col("trg.create_date"),
        F.col("trg.update_date")
    )

    df_existing = df_join.filter(F.col(surrogate_key).isNotNull())
    df_new = df_join.filter(F.col(surrogate_key).isNull())

In [0]:
# ==============================
# HEDEF TABLOYA YAZMA
# ==============================
# unionByName'de allowMissingColumns parametresini kullanın
key_cols = []  # Fix: remove empty string, avoids unresolved column error

df_final = df_existing_final.unionByName(df_new_final, allowMissingColumns=True)
if target_exists:
    # Mevcut tabloyu güncelle
    df_final.write \
        .mode("overwrite") \
        .option("overwriteSchema", "false") \
        .saveAsTable(full_target)
else:
    # İlk kez oluştur
    df_final.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_target)

print(f"Veri başarıyla {full_target} tablosuna kaydedildi.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Veri başarıyla workspace.gold.gold_waffle tablosuna kaydedildi.
